# CS3631 Project — Baseline: Spatio-Temporal GNN for Dengue Forecasting (Sri Lanka, 25 districts)

**Baseline models:** GCN and GAT (the mandated graph baselines) for multi-horizon weekly dengue incidence forecasting.

This notebook is **self-contained** and uses only **PyTorch Geometric** (not `torch-geometric-temporal`, which is fragile on Colab). It matches the data conventions of the reference repo `disease_modeling_MLOS2` exactly, so our numbers are comparable to the Weng et al. baseline:

- **Data:** `sri_lanka_2013-2022_shifted.npy`, shape `(459 weeks, 25 districts, 11 features)`. Target = weekly **cases** (feature index `5`, i.e. `-6`).
- **Graph:** `sri_lanka_adj_list.json` — 25 districts, geographic adjacency + self-loops → 141 directed edges.
- **Task:** window `W=3` weeks → predict `H=3` weeks ahead (per-district). We report metrics **per horizon** (h=1, h=2, h=3).
- **Split:** chronological 70% train / 10% val / 20% test (no shuffling → no temporal leakage).
- **Metrics:** RMSE, MAE, MAPE (on the original case scale, after inverse z-norm).

> **What you do:** upload the two data files to Google Drive, set `DATA_DIR` in the config cell, then Runtime ▸ Run all.

---
### How to get the two required data files
Both already live in the reference GitHub repo. Easiest path:
1. Download them from GitHub (raw):
   - `sri_lanka_2013-2022_shifted.npy` → `disease_modeling_MLOS2/Data/Datasets/`
   - `sri_lanka_adj_list.json` → `disease_modeling_MLOS2/Models/`
2. Put both in one Google Drive folder, e.g. `MyDrive/dengue_baseline/`.
3. Set `DATA_DIR = "/content/drive/MyDrive/dengue_baseline"` below.

(Cell 2 can also auto-download them straight from GitHub if you'd rather skip Drive.)

## 1. Environment setup
Install PyTorch Geometric wheels matching the Colab-resident torch build.

In [1]:
# Colab already ships torch. Install PyG + its compiled companions against the resident torch build.
import torch, sys, subprocess
TORCH = torch.__version__.split('+')[0]
CUDA  = ('cu' + torch.version.cuda.replace('.', '')) if torch.cuda.is_available() else 'cpu'
print("torch:", torch.__version__, "| wheel tag:", f"{TORCH}+{CUDA}")

# torch-geometric is pure-python; the scatter/sparse companions need the matching wheel index.
def pip(*a): subprocess.run([sys.executable, "-m", "pip", "install", "-q", *a], check=False)

pip("torch-geometric")
pip("torch-scatter", "torch-sparse",
    "-f", f"https://data.pyg.org/whl/torch-{TORCH}+{CUDA}.html")

import torch_geometric
print("torch_geometric:", torch_geometric.__version__)

torch: 2.11.0+cu128 | wheel tag: 2.11.0+cu128
torch_geometric: 2.8.0.post1


## 2. Get the data
Option A mounts Drive; Option B downloads the two files directly from GitHub. Use whichever you prefer.

In [2]:
import os

USE_DRIVE = False   # <-- set False to auto-download from GitHub instead

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = "/content/drive/MyDrive/dengue_baseline"   # <-- EDIT to your folder
else:
    DATA_DIR = "/content/data"
    os.makedirs(DATA_DIR, exist_ok=True)
    base = "https://raw.githubusercontent.com/MLOpenSourceOpenScience/disease_modeling_MLOS2/main"
    import urllib.request
    urllib.request.urlretrieve(f"{base}/Data/Datasets/sri_lanka_2013-2022_shifted.npy",
                               f"{DATA_DIR}/sri_lanka_2013-2022_shifted.npy")
    urllib.request.urlretrieve(f"{base}/Models/sri_lanka_adj_list.json",
                               f"{DATA_DIR}/sri_lanka_adj_list.json")

NPY_PATH = os.path.join(DATA_DIR, "sri_lanka_2013-2022_shifted.npy")
ADJ_PATH = os.path.join(DATA_DIR, "sri_lanka_adj_list.json")
assert os.path.exists(NPY_PATH), f"Missing {NPY_PATH}"
assert os.path.exists(ADJ_PATH), f"Missing {ADJ_PATH}"
print("Found both data files.")

Found both data files.


## 3. Configuration
All knobs in one place — these are the hyperparameters we tune in §8.

In [3]:
import numpy as np, json, random, torch

class CFG:
    npy_path      = NPY_PATH
    adj_path      = ADJ_PATH
    cases_idx     = 5        # target column in the 11-feature array (== -6, matches ref repo)
    n_nodes       = 25
    window        = 3        # input weeks (W)
    horizon       = 3        # forecast weeks ahead (H)
    self_loops    = True
    train_split   = 0.70
    val_split     = 0.10     # test = remaining 0.20
    use_all_feats = True     # True: 11 features per node; False: cases-only (univariate)

    # model
    model         = "GCN"    # "GCN" or "GAT"
    hidden        = 64
    gat_heads     = 8
    dropout       = 0.1

    # optim
    lr            = 1e-3
    weight_decay  = 5e-4
    epochs        = 150
    patience      = 25       # early stopping on val RMSE
    seed          = 0

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

set_seed(CFG.seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cuda


## 4. Load data, build the graph, make windowed samples

We z-normalize using **train-set statistics only** (computed after the chronological cut) to avoid leaking test information into normalization — a small but important correctness fix over normalizing on the whole array.

In [4]:
def load_adjacency(adj_path, n_nodes, self_loops=True):
    adj = json.load(open(adj_path))
    names = sorted(adj.keys())
    idx = {v: i for i, v in enumerate(names)}
    A = np.zeros((n_nodes, n_nodes), dtype=np.float32)
    if self_loops:
        np.fill_diagonal(A, 1.0)
    for d in adj:
        for nb in adj[d]:
            A[idx[d], idx[nb]] = 1.0
    edge_index = np.array([[i, j] for i in range(n_nodes) for j in range(n_nodes) if A[i, j]]).T
    return torch.tensor(edge_index, dtype=torch.long), names

edge_index, DISTRICTS = load_adjacency(CFG.adj_path, CFG.n_nodes, CFG.self_loops)
print("edge_index:", tuple(edge_index.shape), "| districts:", len(DISTRICTS))

# raw array: (weeks, nodes, feats)
raw = np.nan_to_num(np.load(CFG.npy_path, allow_pickle=True)).astype(np.float32)
T, N, F = raw.shape
print("raw:", raw.shape)

# chronological indices for the windowed samples
# sample i uses weeks [i-W, i) as input and [i, i+H) as target -> valid i in [W, T-H)
sample_ids = list(range(CFG.window, T - CFG.horizon))
n_samples  = len(sample_ids)
t_end = int(CFG.train_split * n_samples)
v_end = int((CFG.train_split + CFG.val_split) * n_samples)
print(f"samples: {n_samples} -> train {t_end}, val {v_end-t_end}, test {n_samples-v_end}")

# --- normalization from TRAIN weeks only ---
last_train_week = sample_ids[t_end - 1]          # last input week used by training
train_weeks = raw[:last_train_week]              # weeks strictly before val region
if CFG.use_all_feats:
    feat_mean = train_weeks.reshape(-1, F).mean(0)
    feat_std  = train_weeks.reshape(-1, F).std(0) + 1e-6
else:
    feat_mean = np.zeros(F, np.float32); feat_std = np.ones(F, np.float32)
# target (cases) stats — used to inverse-transform predictions back to case scale
c = CFG.cases_idx
tgt_mean = float(train_weeks[..., c].mean())
tgt_std  = float(train_weeks[..., c].std() + 1e-6)
print(f"target(cases) train mean={tgt_mean:.2f} std={tgt_std:.2f}")

def z(a): return (a - feat_mean) / feat_std
norm = z(raw)                                    # (T,N,F) normalized features
cases_norm = (raw[..., c] - tgt_mean) / tgt_std  # (T,N) normalized target

edge_index: (2, 141) | districts: 25
raw: (459, 25, 11)
samples: 453 -> train 317, val 45, test 91
target(cases) train mean=46.80 std=114.88


### Build tensors of windowed samples
Each sample is a graph snapshot: node features are the last `W` weeks (flattened to `F*W` per node if multivariate, or `W` if cases-only), and the label is the next `H` weekly case values per node.

In [5]:
def make_samples(ids):
    X, Y = [], []
    for i in ids:
        if CFG.use_all_feats:
            # (W, N, F) -> (N, F, W) -> (N, F*W)
            xi = norm[i-CFG.window:i]                     # (W,N,F)
            xi = np.transpose(xi, (1, 2, 0)).reshape(N, F*CFG.window)
        else:
            xi = cases_norm[i-CFG.window:i].T             # (N, W)
        yi = cases_norm[i:i+CFG.horizon].T                # (N, H)  (normalized target)
        X.append(xi); Y.append(yi)
    return torch.tensor(np.stack(X), dtype=torch.float), torch.tensor(np.stack(Y), dtype=torch.float)

train_ids = sample_ids[:t_end]
val_ids   = sample_ids[t_end:v_end]
test_ids  = sample_ids[v_end:]

Xtr, Ytr = make_samples(train_ids)
Xva, Yva = make_samples(val_ids)
Xte, Yte = make_samples(test_ids)
in_dim = Xtr.shape[-1]
print("X train:", tuple(Xtr.shape), "| Y train:", tuple(Ytr.shape), "| in_dim per node:", in_dim)

X train: (317, 25, 33) | Y train: (317, 25, 3) | in_dim per node: 33


## 5. Baseline models: GCN and GAT

A shared **spatial encoder → temporal/MLP head** design. Two graph conv layers aggregate information across neighbouring districts; a linear head maps each node's embedding to `H` future weekly case values. This is the standard, defensible GNN baseline the project mandates, and the clean hook where we will later attach the **physics-informed loss** and **GAN-augmented** training.

In [6]:
import torch.nn as nn
import torch.nn.functional as Fnn
from torch_geometric.nn import GCNConv, GATConv

class GNNBaseline(nn.Module):
    def __init__(self, in_dim, hidden, horizon, kind="GCN", heads=8, dropout=0.1):
        super().__init__()
        self.kind, self.dropout = kind, dropout
        if kind == "GCN":
            self.conv1 = GCNConv(in_dim, hidden)
            self.conv2 = GCNConv(hidden, hidden)
        elif kind == "GAT":
            self.conv1 = GATConv(in_dim, hidden // heads, heads=heads, dropout=dropout)
            self.conv2 = GATConv(hidden, hidden, heads=1, concat=True, dropout=dropout)
        else:
            raise ValueError(kind)
        self.head = nn.Sequential(nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden, horizon))

    def forward(self, x, edge_index):
        h = Fnn.relu(self.conv1(x, edge_index))
        h = Fnn.dropout(h, self.dropout, training=self.training)
        h = Fnn.relu(self.conv2(h, edge_index))
        return self.head(h)              # (N, horizon)

def build_model():
    return GNNBaseline(in_dim, CFG.hidden, CFG.horizon,
                       kind=CFG.model, heads=CFG.gat_heads, dropout=CFG.dropout).to(device)

print(build_model())

GNNBaseline(
  (conv1): GCNConv(33, 64)
  (conv2): GCNConv(64, 64)
  (head): Sequential(
    (0): ReLU()
    (1): Dropout(p=0.1, inplace=False)
    (2): Linear(in_features=64, out_features=3, bias=True)
  )
)


## 6. Metrics and training loop
Metrics are computed on the **original case scale** (inverse z-norm) and reported **per horizon**.

In [7]:
def inv(t):  # inverse z-norm back to case counts
    return t * tgt_std + tgt_mean

def metrics_per_horizon(pred, truth):
    # pred, truth: (samples, N, H) on case scale; returns per-horizon + overall dict
    # NOTE on percentage error: dengue counts are often 0 in low-incidence
    # district-weeks, so plain MAPE (divide by y) explodes. We report SMAPE
    # (symmetric, bounded) as the primary % metric, plus MAPE computed ONLY on
    # weeks with >=1 true case (masked) for comparability with prior work.
    def _stats(p, y):
        rmse = torch.sqrt(torch.mean((p - y) ** 2)).item()
        mae  = torch.mean(torch.abs(p - y)).item()
        smape = torch.mean(2*torch.abs(p - y) / (torch.abs(p)+torch.abs(y)+1e-6)).item() * 100
        mask = y >= 1.0
        mape = (torch.mean(torch.abs(p[mask]-y[mask]) / y[mask]).item()*100) if mask.any() else float('nan')
        return dict(RMSE=rmse, MAE=mae, SMAPE=smape, MAPE=mape)
    out = {}
    H = pred.shape[-1]
    for h in range(H):
        out[f"h{h+1}"] = _stats(pred[..., h], truth[..., h])
    out["overall"] = _stats(pred, truth)
    return out

@torch.no_grad()
def evaluate(model, X, Y):
    model.eval()
    ei = edge_index.to(device)
    preds, truths = [], []
    for i in range(X.shape[0]):
        p = model(X[i].to(device), ei)          # (N,H) normalized
        preds.append(inv(p).cpu()); truths.append(inv(Y[i]))
    return metrics_per_horizon(torch.stack(preds), torch.stack(truths))

def train(model, verbose=True):
    ei = edge_index.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)
    loss_fn = nn.MSELoss()
    best_val, best_state, wait = float("inf"), None, 0
    hist = []
    for ep in range(CFG.epochs):
        model.train(); perm = torch.randperm(Xtr.shape[0]); ep_loss = 0.0
        for i in perm:                          # sample-wise (batch of one graph snapshot)
            opt.zero_grad()
            p = model(Xtr[i].to(device), ei)
            loss = loss_fn(p, Ytr[i].to(device))
            loss.backward(); opt.step(); ep_loss += loss.item()
        val = evaluate(model, Xva, Yva)["overall"]["RMSE"]
        hist.append(val)
        if val < best_val - 1e-4:
            best_val, best_state, wait = val, {k: v.detach().cpu().clone()
                                               for k, v in model.state_dict().items()}, 0
        else:
            wait += 1
        if verbose and (ep % 10 == 0 or ep == CFG.epochs-1):
            print(f"epoch {ep:3d} | train MSE {ep_loss/Xtr.shape[0]:.4f} | val RMSE {val:.3f} | best {best_val:.3f}")
        if wait >= CFG.patience:
            if verbose: print(f"early stop @ epoch {ep} (best val RMSE {best_val:.3f})")
            break
    if best_state: model.load_state_dict(best_state)
    return model, hist

## 7. Train & evaluate the baseline
Runs the model in `CFG.model` (GCN by default). Switch to GAT by setting `CFG.model = 'GAT'`.

In [8]:
set_seed(CFG.seed)
model = build_model()
model, hist = train(model)

def show(name, m):
    print(f"\n=== {name} ===")
    for k in ["h1","h2","h3","overall"]:
        d = m[k]; print(f"{k:8s} RMSE {d['RMSE']:7.3f} | MAE {d['MAE']:7.3f} | SMAPE {d['SMAPE']:6.1f}% | MAPE(>=1) {d['MAPE']:6.1f}%")

test_m = evaluate(model, Xte, Yte)
show(f"{CFG.model} — TEST", test_m)

epoch   0 | train MSE 0.8148 | val RMSE 108.521 | best 108.521
epoch  10 | train MSE 0.3615 | val RMSE 102.368 | best 101.582
epoch  20 | train MSE 0.3074 | val RMSE 99.470 | best 96.565
epoch  30 | train MSE 0.2779 | val RMSE 97.524 | best 95.144
epoch  40 | train MSE 0.2450 | val RMSE 95.676 | best 93.344
epoch  50 | train MSE 0.2455 | val RMSE 103.072 | best 93.242
epoch  60 | train MSE 0.2773 | val RMSE 101.074 | best 93.242
early stop @ epoch 68 (best val RMSE 93.242)

=== GCN — TEST ===
h1       RMSE  73.854 | MAE  24.415 | SMAPE  119.0% | MAPE(>=1)  580.0%
h2       RMSE  71.665 | MAE  24.258 | SMAPE  119.0% | MAPE(>=1)  471.8%
h3       RMSE  69.883 | MAE  23.909 | SMAPE  120.0% | MAPE(>=1)  538.8%
overall  RMSE  71.819 | MAE  24.194 | SMAPE  119.3% | MAPE(>=1)  530.2%


### Persistence sanity-check baseline
A naive "next week = this week" forecast. Any real model must beat this.

In [9]:
@torch.no_grad()
def persistence(X, Y):
    # last observed week's cases, repeated H times. cases is the last W-block feature.
    preds, truths = [], []
    for i in range(X.shape[0]):
        if CFG.use_all_feats:
            # node feature layout is (F,W) flattened; cases row = CFG.cases_idx, last time = W-1
            last = X[i].view(N, F, CFG.window)[:, CFG.cases_idx, -1]
        else:
            last = X[i][:, -1]
        p = inv(last).unsqueeze(-1).repeat(1, CFG.horizon)
        preds.append(p); truths.append(inv(Y[i]))
    return metrics_per_horizon(torch.stack(preds), torch.stack(truths))

show("Persistence — TEST", persistence(Xte, Yte))


=== Persistence — TEST ===
h1       RMSE  58.762 | MAE  12.594 | SMAPE   90.8% | MAPE(>=1)  211.8%
h2       RMSE  59.522 | MAE  13.663 | SMAPE   94.1% | MAPE(>=1)  182.9%
h3       RMSE  60.333 | MAE  15.098 | SMAPE   98.1% | MAPE(>=1)  250.3%
overall  RMSE  59.542 | MAE  13.785 | SMAPE   94.3% | MAPE(>=1)  215.0%


## 8. Hyperparameter tuning (required for Phase 1)

Grid search over learning rate, hidden size, and dropout, **selecting on validation RMSE** (never test). We report the best config and its test metrics. This satisfies the proposal's "baseline experimental results including hyperparameter tuning" requirement.

> Keep the grid small on Colab CPU; expand if you have GPU. Each cell run is logged so you can paste the table straight into the paper.

In [ ]:
import itertools, pandas as pd

grid = {
    "lr":      [1e-2, 1e-3, 5e-4],
    "hidden":  [32, 64],
    "dropout": [0.0, 0.1, 0.3],
}
MODEL_TO_TUNE = "GCN"      # rerun with "GAT" for the GAT baseline table

rows = []
combos = list(itertools.product(*grid.values()))
print(f"{len(combos)} configs to try for {MODEL_TO_TUNE}...")
for lr, hid, dp in combos:
    CFG.model, CFG.lr, CFG.hidden, CFG.dropout = MODEL_TO_TUNE, lr, hid, dp
    set_seed(CFG.seed)
    m = build_model()
    m, _ = train(m, verbose=False)
    val = evaluate(m, Xva, Yva)["overall"]
    tst = evaluate(m, Xte, Yte)["overall"]
    rows.append(dict(model=MODEL_TO_TUNE, lr=lr, hidden=hid, dropout=dp,
                     val_RMSE=round(val["RMSE"],3),
                     test_RMSE=round(tst["RMSE"],3),
                     test_MAE=round(tst["MAE"],3),
                     test_SMAPE=round(tst["SMAPE"],1)))
    print(f"lr={lr:<6} hid={hid:<3} dp={dp:<3} -> val RMSE {val['RMSE']:.3f} | test RMSE {tst['RMSE']:.3f}")

df = pd.DataFrame(rows).sort_values("val_RMSE").reset_index(drop=True)
print("\nBest config (by val RMSE):")
print(df.head(1).to_string(index=False))
df

18 configs to try for GCN...
lr=0.01   hid=32  dp=0.0 -> val RMSE 91.668 | test RMSE 66.080
lr=0.01   hid=32  dp=0.1 -> val RMSE 100.193 | test RMSE 56.030
lr=0.01   hid=32  dp=0.3 -> val RMSE 95.966 | test RMSE 59.615
lr=0.01   hid=64  dp=0.0 -> val RMSE 95.845 | test RMSE 60.106
lr=0.01   hid=64  dp=0.1 -> val RMSE 101.028 | test RMSE 53.453
lr=0.01   hid=64  dp=0.3 -> val RMSE 108.140 | test RMSE 54.951
lr=0.001  hid=32  dp=0.0 -> val RMSE 92.377 | test RMSE 80.380
lr=0.001  hid=32  dp=0.1 -> val RMSE 92.923 | test RMSE 69.391
lr=0.001  hid=32  dp=0.3 -> val RMSE 93.850 | test RMSE 65.460
lr=0.001  hid=64  dp=0.0 -> val RMSE 90.871 | test RMSE 67.169
lr=0.001  hid=64  dp=0.1 -> val RMSE 91.597 | test RMSE 70.204
lr=0.001  hid=64  dp=0.3 -> val RMSE 93.079 | test RMSE 67.054
lr=0.0005 hid=32  dp=0.0 -> val RMSE 92.363 | test RMSE 70.716
lr=0.0005 hid=32  dp=0.1 -> val RMSE 93.541 | test RMSE 79.081
lr=0.0005 hid=32  dp=0.3 -> val RMSE 96.082 | test RMSE 63.554
lr=0.0005 hid=64  dp=0.

### Lock in the best config and report final per-horizon results

In [ ]:
best = df.iloc[0]
CFG.model, CFG.lr, CFG.hidden, CFG.dropout = best["model"], float(best["lr"]), int(best["hidden"]), float(best["dropout"])
print("Best:", dict(model=CFG.model, lr=CFG.lr, hidden=CFG.hidden, dropout=CFG.dropout))
set_seed(CFG.seed)
final = build_model(); final, hist = train(final, verbose=False)
show(f"FINAL {CFG.model} (tuned) — TEST", evaluate(final, Xte, Yte))

## 9. Visualize predictions
Observed vs predicted weekly cases (1-week-ahead head) for a chosen district on the test set.

In [ ]:
import matplotlib.pyplot as plt

@torch.no_grad()
def predict_series(model, X, horizon_step=0):
    ei = edge_index.to(device); preds = []
    model.eval()
    for i in range(X.shape[0]):
        p = inv(model(X[i].to(device), ei)).cpu()
        preds.append(p[:, horizon_step])
    return torch.stack(preds)            # (samples, N)

district = "Colombo"
d = DISTRICTS.index(district)
pred_series  = predict_series(final, Xte, 0)[:, d].numpy()
truth_series = torch.stack([inv(Yte[i]) for i in range(Xte.shape[0])])[:, d, 0].numpy()

plt.figure(figsize=(11,4))
plt.plot(truth_series, label="Observed", linewidth=1.6)
plt.plot(pred_series, "--", label=f"Predicted ({CFG.model}, 1-wk)", linewidth=1.4)
plt.title(f"Dengue cases — {district} (test set, 1-week-ahead)")
plt.xlabel("Test week"); plt.ylabel("Cases"); plt.legend(); plt.tight_layout(); plt.show()

## 10. Where the contributions plug in (roadmap, not for Phase 1 run)

This baseline is deliberately structured so each novelty attaches at a clean seam:

1. **Physics-informed loss** → modify `train()`'s loss to `MSE + λ_phys · L_residual`, where `L_residual` penalizes deviation from an **SEIR-SEI host–vector** update (population conservation + neighbour smoothness). Nothing else changes.
2. **GAN augmentation** → generate synthetic `(window → horizon)` sequences (TimeGAN / conditional GAN), prepend them to `Xtr/Ytr`, keep the identical model and eval. Report the ablation *with vs without* augmentation.
3. **Architecture** → swap the two `GCNConv`/`GATConv` layers for a learned/adaptive adjacency (Graph WaveNet-style) — the encoder interface stays the same.

Keep this baseline notebook frozen as the reference point; branch a copy per contribution so the ablation table is apples-to-apples.

---
### Reporting checklist for the proposal (Phase 1)
- [ ] GCN baseline: per-horizon RMSE/MAE/MAPE (§7)
- [ ] GAT baseline: rerun §7–8 with `CFG.model='GAT'`
- [ ] Persistence baseline (must be beaten) (§7)
- [ ] Hyperparameter tuning table (§8)
- [ ] One prediction plot (§9)